In [ ]:
#Computes confusion matrix totals (TP, TN, FP, FN) and derives accuracy, precision, recall, F1 score, and Dice score

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

def compute_confusion_metrics(model, test_loader, device):

    model.eval()

    # Initialize global counters for confusion matrix totals
    total_TP = 0
    total_TN = 0
    total_FP = 0
    total_FN = 0

    loop_stats = tqdm(test_loader, desc="Evaluating")
    for batch in loop_stats:
        data, target, _ = batch # Unpack batch
        data = data.to(device)
        target = target.to(device)
        prediction = model(data)

        # Convert tensors to NumPy arrays
        pred_np = prediction.detach().cpu().squeeze().numpy()
        gt_np   = target.detach().cpu().squeeze().numpy()

        # Apply threshold
        pred_bin = (pred_np > 0.5).astype(np.uint8)
        gt_bin   = (gt_np > 0.5).astype(np.uint8)

        # Compute confusion matrix
        TP = np.sum((pred_bin == 1) & (gt_bin == 1))
        TN = np.sum((pred_bin == 0) & (gt_bin == 0))
        FP = np.sum((pred_bin == 1) & (gt_bin == 0))
        FN = np.sum((pred_bin == 0) & (gt_bin == 1))

        # Accumulate totals across all test images
        total_TP += TP
        total_TN += TN
        total_FP += FP
        total_FN += FN

    # Summarize confusion matrix 
    df_conf = pd.DataFrame({
        "Metric": ["True Positive", "True Negative", "False Positive", "False Negative"],
        "Count": [total_TP, total_TN, total_FP, total_FN]
    })

    # avoid division by zero
    eps = 1e-8

    # Compute evaluation metrics
    accuracy  = (total_TP + total_TN) / (total_TP + total_TN + total_FP + total_FN + eps)
    precision = total_TP / (total_TP + total_FP + eps)
    recall    = total_TP / (total_TP + total_FN + eps)
    f1_score  = 2 * (precision * recall) / (precision + recall + eps)
    dice      = (2 * total_TP) / (2 * total_TP + total_FP + total_FN + eps)

    # Create dictionary
    metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1_score,
        "dice": dice
    }

    return df_conf, metrics

In [ ]:
# Execution of Evaluation

if __name__ == "__main__":

    # Select device 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load trained model
    from main_training_script import UNetModel   # adjust import to your project structure

    model = UNetModel(in_channels=3, out_channels=1).to(device)

    model_path = "models/epoch_49.pth"   # local path to epoch
    print(f"Loading model from: {model_path}")

    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict["net_state_dict"])

    # Load your test dataset
    from provider.dataset_provider import get_loader   # adjust import to your project structure

    test_loader = get_loader(
        base_path="data_split",   # local path to split dataset
        dataset_type="test",
        batch_size=1
    )

    # Run evaluation
    df_conf, metrics = compute_confusion_metrics(model, test_loader, device)

    print("\nConfusion Matrix Totals:")
    print(df_conf)

    print("\nMetrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.6f}")